# PRAGma App — LOT 기반 에칭 공정 분석 시스템

**파이프라인:**
1. LOT 번호 입력 → MES(CSV Mock)에서 공정 데이터 자동 조회
2. LightGBM 모델로 에칭 속도 예측
3. Claude LLM + RAG 지식베이스로 공정 해석 및 조치 방안 제시

**로컬 실행 시:** `ANTHROPIC_API_KEY` 환경변수 필요  
**Google Colab 실행 시:** 마지막 셀에서 Drive 마운트 후 경로 수정

## 0. 환경 설정

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

try:
    import anthropic
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic'], check=True)
    import anthropic

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# 로컬: PRAGma 프로젝트 루트 기준
# Colab: /content/PRAGma 로 변경
BASE_DIR = Path(".")  # pragma_app.ipynb 가 PRAGma 루트에 있다고 가정

MODEL_PATH      = BASE_DIR / "notebooks/best_LightGBM_mass_speed_regressor.pkl"
CSV_PATH        = BASE_DIR / "data/Train_0319.csv"
PAPER_JSON_PATH = BASE_DIR / "data/rag_data_all.json"
SHAP_MD_PATH    = BASE_DIR / "notebooks/shap_analysis_for_rag.md"

# Anthropic API Key (환경변수에서 읽거나 직접 입력)
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = input("Anthropic API Key를 입력하세요: ").strip()

print("설정 완료")

## 1. 모델 및 데이터 로드

In [ ]:
# LightGBM 모델 로드
with open(MODEL_PATH, "rb") as f:
    lgbm_model = pickle.load(f)

MODEL_FEATURES = lgbm_model.feature_name_
print(f"모델 로드 완료 — 피처 수: {len(MODEL_FEATURES)}")
print(f"피처 목록: {MODEL_FEATURES[:8]} ...")

# MES Mock 데이터 로드 (실제 환경에서는 MES API 호출로 대체)
df_mes = pd.read_csv(CSV_PATH, encoding="cp949")
print(f"\nMES Mock 데이터 로드 완료 — LOT 수: {len(df_mes)}, 컬럼 수: {len(df_mes.columns)}")
print(f"LOT 예시: {df_mes['LOT'].head(5).tolist()}")

# RAG 지식베이스 로드
with open(PAPER_JSON_PATH, "r", encoding="utf-8") as f:
    paper_rules = json.load(f)

shap_content = SHAP_MD_PATH.read_text(encoding="utf-8") if SHAP_MD_PATH.exists() else ""

print(f"\nRAG 지식베이스 로드 완료")
print(f"  논문 rule 수: {len(paper_rules)}")
print(f"  SHAP 분석 MD 존재: {bool(shap_content)}")

## 2. CSV 컬럼 → 모델 피처 매핑

실제 MES에서 받는 raw 컬럼명을 LightGBM 모델 입력 피처명으로 변환합니다.

In [ ]:
# ── 연속형 컬럼 매핑 (동적 탐색) ─────────────────────────────────────────────
# Cu 표면두께 컬럼 (고정명)
CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val"   : "cu_thick_max",
    "Cu 표면두께 AVG_VAL"   : "cu_thick_avg",
    "Cu 표면두께 Min_Val"   : "cu_thick_min",
    "Cu 표면두께 Std_Val"   : "cu_thick_std",
    "Cu 표면두께 Median_Val": "cu_thick_median",
}

# 분析치_ 컬럼은 인코딩에 따라 '析치'가 다를 수 있어 suffix 기반으로 동적 탐색
_feat_suffix_map = {
    "Etch factor"             : "etch_factor",
    "Etching(염화동) - Cu"    : "meas_etch_cu",
    "Etching(염화동) - HCl"   : "meas_etch_hcl",
    "Etching(염화동) - 비중"  : "meas_etch_sg",
    "Etching(염화동) - 온도"  : "meas_etch_temp",
    "Etching-첨가제(HB-120EF)": "meas_etch_additive",
    "Etching량"               : "meas_etch_amount",
    "Soft Etch - Cu"          : "meas_softetch_cu",
    "Soft Etch - H2SO4"       : "meas_softetch_h2so4",
    "Soft Etch - SPS"         : "meas_softetch_sps",
    "박리액 - 농도"           : "meas_strip_conc",
    "수세수 - pH"             : "meas_rinse_ph",
    "현상액 - pH"             : "meas_dev_ph",
    "현상액 - 농도"           : "meas_dev_conc",
}

# df.columns에서 '析치' 포함 컬럼을 찾아 suffix로 매핑 (인코딩 차이 자동 대응)
for col in df_mes.columns:
    if '析치' in col or '분析치' in col or '분석치' in col:
        suffix = col.split('_', 1)[-1] if '_' in col else col
        if suffix in _feat_suffix_map:
            CONTINUOUS_MAP[col] = _feat_suffix_map[suffix]

# 하위 호환 alias
CONTINUOUS_MAP_RESOLVED = CONTINUOUS_MAP

print(f"매핑 완료: {len(CONTINUOUS_MAP_RESOLVED)}개 연속형 피처")
for k, v in CONTINUOUS_MAP_RESOLVED.items():
    print(f"  {k!r:45s} → {v}")

## 3. MES Mock 함수

실제 환경에서는 MES REST API 호출로 교체합니다.

In [ ]:
def get_lot_data(lotno: str) -> dict | None:
    """MES Mock: LOT 번호 → 공정 데이터 딕셔너리 반환
    
    실제 MES 연동 시 이 함수만 교체하면 됩니다:
        response = requests.get(f"{MES_URL}/lot/{lotno}")
        return response.json()
    """
    rows = df_mes[df_mes["LOT"] == lotno]
    if rows.empty:
        print(f"[경고] LOT '{lotno}' 를 찾을 수 없습니다.")
        print(f"  사용 가능한 LOT 예시: {df_mes['LOT'].head(10).tolist()}")
        return None
    return rows.iloc[0].to_dict()


# 테스트
sample = get_lot_data("A20000")
if sample:
    print(f"LOT A20000 조회 성공")
    print(f"  실제 에칭 속도: {sample.get('부식 Speed')} m/min")
    print(f"  에칭 온도: {sample.get('분析치_Etching(염화동) - 온도')} °C")
    print(f"  에칭 비중: {sample.get('분析치_Etching(염화동) - 비중')}")

## 4. 피처 변환 (MES 데이터 → LightGBM 입력)

In [ ]:
# ── 범주형 컬럼 정의 ───────────────────────────────────────────────────────────
CATEGORICAL_COLS = {
    "재작업사유" : {
        "prefix": "rework_history",
        "values": ["Unknown", "기타", "기판 겹침", "두께 미달", "딤플", "설비 에러"],
        "sep"   : "_",   # 모델은 공백을 언더스코어로 저장했음
    },
    "노광 설비정보": {
        "prefix": "expo_eq_id",
        "values": [f"EXP-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
    "DES 설비정보": {
        "prefix": "des_eq_id",
        "values": [f"DES-{i:03d}" for i in range(1, 7)],
        "sep"   : "_",
    },
    "정면 설비정보": {
        "prefix": "brush_eq_id",
        "values": [f"PRE-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
}


def prepare_features(lot_data: dict) -> pd.DataFrame:
    """MES 딕셔너리 → LightGBM 입력 DataFrame (1행)"""
    row = {}

    # 1. 연속형 피처
    for csv_col, feat_name in CONTINUOUS_MAP_RESOLVED.items():
        val = lot_data.get(csv_col, np.nan)
        row[feat_name] = float(val) if val is not None and str(val) not in ("", "nan", "None") else np.nan

    # 2. 범주형 → one-hot
    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw_val = str(lot_data.get(csv_col, "")).strip()
        # 공백을 언더스코어로 변환 (rework_history 한정)
        norm_val = raw_val.replace(" ", "_") if cfg["prefix"] == "rework_history" else raw_val
        for v in cfg["values"]:
            norm_v = v.replace(" ", "_") if cfg["prefix"] == "rework_history" else v
            feat_key = f"{cfg['prefix']}{cfg['sep']}{norm_v}"
            row[feat_key] = 1 if norm_val == norm_v else 0

    # 3. 모델이 요구하는 피처 순서로 정렬, 없는 피처는 0 채움
    for feat in MODEL_FEATURES:
        if feat not in row:
            row[feat] = 0

    return pd.DataFrame([row])[MODEL_FEATURES]


# 테스트
if sample:
    X = prepare_features(sample)
    print(f"피처 변환 완료: shape = {X.shape}")
    print(X[[
        "cu_thick_avg", "etch_factor", "meas_etch_temp",
        "meas_etch_sg", "meas_etch_cu", "expo_eq_id_EXP-001"
    ]].to_string(index=False))

## 5. ML 예측 (LightGBM)

In [ ]:
def predict_etch_speed(lot_data: dict) -> tuple[float, pd.DataFrame]:
    """LightGBM으로 에칭 속도 예측
    
    Returns:
        (예측값, 피처 DataFrame)
    """
    X = prepare_features(lot_data)
    pred = lgbm_model.predict(X)[0]
    return float(pred), X


def get_opls_bounds() -> dict:
    """에칭 공정 OPLS 기준값 (하드코딩 — 실제는 MES에서 조회)"""
    return {
        "meas_etch_temp"    : {"lcl": 44.5, "sl": 48.0, "ucl": 53.0, "unit": "°C",  "name": "에칭 온도"},
        "meas_etch_sg"      : {"lcl": 1.32,  "sl": 1.37,  "ucl": 1.42,  "unit": "",    "name": "에칭 비중"},
        "meas_etch_cu"      : {"lcl": 125.0, "sl": 155.0, "ucl": 185.0, "unit": "g/L", "name": "에칭 Cu 농도"},
        "meas_etch_hcl"     : {"lcl": 0.3,   "sl": 0.5,   "ucl": 0.7,   "unit": "N",   "name": "에칭 HCl"},
        "meas_etch_additive": {"lcl": 2.6,   "sl": 3.0,   "ucl": 3.4,   "unit": "g/L", "name": "에칭 첨가제"},
    }


def check_opls_status(X: pd.DataFrame) -> list[dict]:
    """OPLS 기준 대비 이탈 항목 체크"""
    bounds = get_opls_bounds()
    alerts = []
    for feat, lim in bounds.items():
        if feat not in X.columns:
            continue
        val = X[feat].iloc[0]
        if pd.isna(val):
            continue
        status = "정상"
        if val > lim["ucl"]:
            status = "UCL 초과 (상한 이탈)"
        elif val < lim["lcl"]:
            status = "LCL 미달 (하한 이탈)"
        elif val > lim["sl"] * 1.02:
            status = "SL 상향 근접"
        elif val < lim["sl"] * 0.98:
            status = "SL 하향 근접"
        alerts.append({
            "피처"  : feat,
            "항목"  : lim["name"],
            "현재값": round(val, 4),
            "LCL"   : lim["lcl"],
            "SL"    : lim["sl"],
            "UCL"   : lim["ucl"],
            "단위"  : lim["unit"],
            "상태"  : status,
        })
    return alerts


# 테스트
if sample:
    pred_speed, X = predict_etch_speed(sample)
    actual_speed  = sample.get("부식 Speed", "N/A")
    print(f"=== LOT: {sample['LOT']} ===")
    print(f"예측 에칭 속도 : {pred_speed:.4f} m/min")
    print(f"실제 에칭 속도 : {actual_speed} m/min")
    if isinstance(actual_speed, (int, float)):
        err = abs(pred_speed - actual_speed) / actual_speed * 100
        print(f"오차율          : {err:.2f}%")
    
    print("\n=== OPLS 상태 ===" )
    alerts = check_opls_status(X)
    df_alerts = pd.DataFrame(alerts)
    print(df_alerts.to_string(index=False))

## 6. RAG 지식베이스 준비

In [ ]:
def build_knowledge_context(max_rules: int = 5) -> str:
    """RAG 지식베이스를 LLM 시스템 프롬프트용 텍스트로 변환"""
    parts = []

    # 논문 기반 공정 rule (상위 N개)
    if paper_rules:
        parts.append("[논문 기반 공정 규칙]")
        for item in paper_rules[:max_rules]:
            parts.append(json.dumps(item, ensure_ascii=False, indent=2))

    # SHAP 분석 결과
    if shap_content:
        parts.append("\n[SHAP 모델 해석 분석]")
        parts.append(shap_content[:3000])  # 토큰 절약

    return "\n\n".join(parts)


def retrieve_relevant_rules(question: str, lot_data: dict, top_k: int = 3) -> str:
    """키워드 기반 관련 rule 검색 (경량 RAG)"""
    keywords = []
    q_lower  = question.lower()

    # 질문 키워드 추출
    keyword_map = {
        "온도"   : ["온도", "temperature", "temp"],
        "비중"   : ["비중", "density", "sg"],
        "속도"   : ["속도", "speed", "컨베이어"],
        "Cu"     : ["cu", "구리", "copper"],
        "불량"   : ["불량", "defect", "이상", "문제"],
        "수율"   : ["수율", "yield"],
        "SHAP"   : ["shap", "중요도", "영향"],
        "과에칭" : ["과에칭", "over-etch", "선폭"],
        "잔동"   : ["잔동", "under-etch"],
    }
    for key, terms in keyword_map.items():
        if any(t in q_lower for t in terms):
            keywords.append(key)

    # paper_rules에서 키워드 매칭
    scored = []
    for item in paper_rules:
        text  = json.dumps(item, ensure_ascii=False).lower()
        score = sum(1 for kw in keywords if kw.lower() in text)
        if score > 0:
            scored.append((score, item))

    scored.sort(key=lambda x: -x[0])
    top_rules = [json.dumps(item, ensure_ascii=False, indent=2) for _, item in scored[:top_k]]

    if not top_rules and paper_rules:
        top_rules = [json.dumps(paper_rules[0], ensure_ascii=False, indent=2)]

    result = "\n---\n".join(top_rules)
    if shap_content:
        result += "\n\n[SHAP 분석]\n" + shap_content[:2000]
    return result


# 테스트
ctx = retrieve_relevant_rules("에칭 온도가 높을 때 문제가 뭐가 있나요?", {})
print(f"검색된 컨텍스트 길이: {len(ctx)} 자")
print(ctx[:500])

## 7. Claude LLM 통합 (RAG + ML 예측 결과 기반 Q&A)

In [ ]:
def build_lot_summary(lot_data: dict, pred_speed: float, X: pd.DataFrame, alerts: list) -> str:
    """LOT 공정 현황 요약 문자열 생성"""
    def safe_get(key, digits=4):
        v = lot_data.get(key)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "N/A"
        return round(v, digits) if isinstance(v, float) else v

    # OPLS 이탈 항목 문자열
    alert_lines = []
    for a in alerts:
        if a["상태"] != "정상":
            alert_lines.append(
                f"  ⚠ {a['항목']}: {a['현재값']}{a['단위']} [{a['상태']}] "
                f"(LCL={a['LCL']}, SL={a['SL']}, UCL={a['UCL']})"
            )
    alert_str = "\n".join(alert_lines) if alert_lines else "  이탈 항목 없음"

    # 에칭 분析치 컬럼 자동 탐색
    etch_cols = {
        "에칭 Cu 농도"  : safe_get(next((c for c in lot_data if 'Etching' in c and 'Cu' in c and '析치' in c), ""), 2),
        "에칭 HCl"      : safe_get(next((c for c in lot_data if 'HCl' in c and '析치' in c), ""), 3),
        "에칭 비중"      : safe_get(next((c for c in lot_data if '비중' in c and '析치' in c), ""), 4),
        "에칭 온도"      : safe_get(next((c for c in lot_data if '온도' in c and '析치' in c), ""), 2),
        "에칭 첨가제"    : safe_get(next((c for c in lot_data if '첨가제' in c and '析치' in c), ""), 3),
        "에칭량"         : safe_get(next((c for c in lot_data if 'Etching량' in c and '析치' in c), ""), 2),
    }

    etch_str = "\n".join(f"  {k}: {v}" for k, v in etch_cols.items())

    return f"""LOT 번호     : {lot_data.get('LOT')}
제품 코드    : {safe_get('통합코드')}
거래처       : {safe_get('거래처')}
공법 구분    : {safe_get('공법구분')}
LAYER        : {safe_get('LAYER')}
DRY FILM     : {safe_get('DRY FILM 정보')}

[에칭 공정 실측치]
{etch_str}
  Cu 표면두께 평균: {safe_get('Cu 표면두께 AVG_VAL')}
  Etch Factor    : {safe_get('분析치_Etch factor') if '분析치_Etch factor' in lot_data else 'N/A'}

[ML 예측 결과]
  예측 에칭 속도 : {pred_speed:.4f} m/min
  실제 에칭 속도 : {safe_get('부식 Speed')} m/min
  검사 결과      : {safe_get('Result Ng2')}

[OPLS 공정 기준 대비 상태]
{alert_str}
"""


def ask_pragma(
    question : str,
    lot_data : dict,
    pred_speed: float,
    X        : pd.DataFrame,
    alerts   : list,
    verbose  : bool = True,
) -> str:
    """Claude API를 사용해 LOT 기반 공정 Q&A 수행"""
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    lot_summary   = build_lot_summary(lot_data, pred_speed, X, alerts)
    rag_context   = retrieve_relevant_rules(question, lot_data)

    system_prompt = f"""당신은 PCB 에칭 공정 전문가 AI입니다.
아래 공정 지식베이스(논문 rule + SHAP 분석)를 참고하여 답변하세요.

=== 공정 지식베이스 ===
{rag_context}
========================

답변 형식:
1. 핵심 진단 (ML 예측 + OPLS 이탈 기반)
2. 관련 공정 변수 분석
3. 근거 (문헌/모델)
4. 조치 방향"""

    user_message = f"""다음 LOT의 공정 현황을 분석하고 질문에 답변해 주세요.

=== 현재 LOT 공정 현황 ===
{lot_summary}
==========================

질문: {question}"""

    if verbose:
        print(f"[질문]: {question}")
        print("LLM 호출 중...")

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}],
    )
    answer = response.content[0].text

    if verbose:
        print("\n[답변]:")
        print(answer)

    return answer


print("LLM 함수 정의 완료")

## 8. 통합 실행 함수

In [ ]:
def analyze_lot(lotno: str, question: str) -> str:
    """LOT 번호 + 질문 → 공정 분석 답변 (메인 엔트리포인트)
    
    실제 MES 연동 시 get_lot_data() 내부만 교체하면 됩니다.
    """
    print(f"\n{'='*60}")
    print(f" LOT {lotno} 분석 시작")
    print(f"{'='*60}")

    # Step 1: MES에서 공정 데이터 조회
    lot_data = get_lot_data(lotno)
    if lot_data is None:
        return f"LOT '{lotno}' 를 찾을 수 없습니다."

    # Step 2: LightGBM으로 에칭 속도 예측
    pred_speed, X = predict_etch_speed(lot_data)
    print(f"▶ 예측 에칭 속도: {pred_speed:.4f} m/min  "
          f"(실제: {lot_data.get('부식 Speed', 'N/A')} m/min)")

    # Step 3: OPLS 기준 이탈 체크
    alerts = check_opls_status(X)
    abnormal = [a for a in alerts if a["상태"] != "정상"]
    if abnormal:
        print(f"▶ OPLS 이탈 항목: {len(abnormal)}개")
        for a in abnormal:
            print(f"   - {a['항목']}: {a['현재값']}{a['단위']} [{a['상태']}]")
    else:
        print("▶ OPLS 이탈 항목: 없음")

    # Step 4: Claude LLM + RAG로 답변 생성
    print()
    answer = ask_pragma(question, lot_data, pred_speed, X, alerts, verbose=True)
    return answer


print("analyze_lot() 함수 정의 완료")

## 9. 테스트 실행

LOT 번호와 질문을 바꿔가며 실행하세요.

In [ ]:
# ── 여기를 수정하세요 ─────────────────────────────────────────────────────────
LOT_NO   = "A20000"
QUESTION = "이 lot의 에칭 공정 상태를 분석하고, 예측 속도와 실제 속도 차이의 원인 및 개선 방향을 설명해 주세요."
# ─────────────────────────────────────────────────────────────────────────────

answer = analyze_lot(LOT_NO, QUESTION)

In [ ]:
# 다른 질문 예시
LOT_NO2   = "A20001"
QUESTION2 = "에칭 온도와 비중이 OPLS 기준에서 어느 정도 이탈해 있으며, 과에칭 위험도는 어떻게 판단하나요?"

answer2 = analyze_lot(LOT_NO2, QUESTION2)

## 10. 배치 분석 (여러 LOT 한 번에 처리)

In [ ]:
def batch_predict(lot_list: list[str]) -> pd.DataFrame:
    """여러 LOT의 예측 결과 및 OPLS 상태를 DataFrame으로 반환 (LLM 호출 없음)"""
    results = []
    for lotno in lot_list:
        lot_data = get_lot_data(lotno)
        if lot_data is None:
            results.append({"LOT": lotno, "예측속도": None, "실제속도": None, "이탈항목": "NOT FOUND"})
            continue

        pred, X   = predict_etch_speed(lot_data)
        actual    = lot_data.get("부식 Speed")
        alerts    = check_opls_status(X)
        abnormal  = [a["항목"] for a in alerts if a["상태"] != "정상"]

        results.append({
            "LOT"        : lotno,
            "예측속도"   : round(pred, 4),
            "실제속도"   : actual,
            "오차율(%)"  : round(abs(pred - actual) / actual * 100, 2) if isinstance(actual, (int, float)) else None,
            "검사결과"   : lot_data.get("Result Ng2"),
            "OPLS이탈"   : ", ".join(abnormal) if abnormal else "정상",
        })

    return pd.DataFrame(results)


# 처음 10개 LOT 배치 분석
sample_lots = df_mes["LOT"].head(10).tolist()
df_batch = batch_predict(sample_lots)
print(df_batch.to_string(index=False))

## 11. Streamlit 앱 코드 생성

아래 셀을 실행하면 `pragma_streamlit.py` 파일이 생성됩니다.  
`streamlit run pragma_streamlit.py` 로 실행하세요.

In [ ]:
STREAMLIT_CODE = '''
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
import streamlit as st
import anthropic
from pathlib import Path

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
BASE_DIR = Path(__file__).parent

# ── 모델/데이터 로드 (st.cache_resource로 1회만 로드) ─────────────────────────
@st.cache_resource
def load_resources():
    with open(BASE_DIR / "notebooks/best_LightGBM_mass_speed_regressor.pkl", "rb") as f:
        model = pickle.load(f)
    df = pd.read_csv(BASE_DIR / "data/Train_0319.csv", encoding="cp949")
    with open(BASE_DIR / "data/rag_data_all.json", "r", encoding="utf-8") as f:
        rules = json.load(f)
    shap_md = BASE_DIR / "notebooks/shap_analysis_for_rag.md"
    shap    = shap_md.read_text(encoding="utf-8") if shap_md.exists() else ""
    return model, df, rules, shap


lgbm_model, df_mes, paper_rules, shap_content = load_resources()
MODEL_FEATURES = lgbm_model.feature_name_

CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val"             : "cu_thick_max",
    "Cu 표면두께 AVG_VAL"             : "cu_thick_avg",
    "Cu 표면두께 Min_Val"             : "cu_thick_min",
    "Cu 표면두께 Std_Val"             : "cu_thick_std",
    "Cu 표면두께 Median_Val"          : "cu_thick_median",
}

# 분析치 컬럼 자동 매핑
_feat_suffix_map = {
    "Etch factor"        : "etch_factor",
    "Etching(염화동) - Cu"   : "meas_etch_cu",
    "Etching(염화동) - HCl"  : "meas_etch_hcl",
    "Etching(염화동) - 비중" : "meas_etch_sg",
    "Etching(염화동) - 온도" : "meas_etch_temp",
    "Etching-첨가제(HB-120EF)": "meas_etch_additive",
    "Etching량"          : "meas_etch_amount",
    "Soft Etch - Cu"     : "meas_softetch_cu",
    "Soft Etch - H2SO4"  : "meas_softetch_h2so4",
    "Soft Etch - SPS"    : "meas_softetch_sps",
    "박리액 - 농도"       : "meas_strip_conc",
    "수세수 - pH"         : "meas_rinse_ph",
    "현상액 - pH"         : "meas_dev_ph",
    "현상액 - 농도"       : "meas_dev_conc",
}
for col in df_mes.columns:
    if "析치" in col or "분析치" in col or "분석치" in col:
        suffix = col.split("_", 1)[-1] if "_" in col else col
        if suffix in _feat_suffix_map:
            CONTINUOUS_MAP[col] = _feat_suffix_map[suffix]

CATEGORICAL_COLS = {
    "재작업사유"   : {"prefix": "rework_history", "values": ["Unknown","기타","기판 겹침","두께 미달","딤플","설비 에러"], "sep": "_"},
    "노광 설비정보": {"prefix": "expo_eq_id",     "values": [f"EXP-{i:03d}" for i in range(1,8)],  "sep": "_"},
    "DES 설비정보" : {"prefix": "des_eq_id",      "values": [f"DES-{i:03d}" for i in range(1,7)],  "sep": "_"},
    "정면 설비정보": {"prefix": "brush_eq_id",    "values": [f"PRE-{i:03d}" for i in range(1,8)],  "sep": "_"},
}

OPLS_BOUNDS = {
    "meas_etch_temp"    : {"lcl": 44.5, "sl": 48.0, "ucl": 53.0, "unit": "°C",  "name": "에칭 온도"},
    "meas_etch_sg"      : {"lcl": 1.32,  "sl": 1.37,  "ucl": 1.42,  "unit": "",    "name": "에칭 비중"},
    "meas_etch_cu"      : {"lcl": 125.0, "sl": 155.0, "ucl": 185.0, "unit": "g/L", "name": "에칭 Cu 농도"},
    "meas_etch_hcl"     : {"lcl": 0.3,   "sl": 0.5,   "ucl": 0.7,   "unit": "N",   "name": "에칭 HCl"},
    "meas_etch_additive": {"lcl": 2.6,   "sl": 3.0,   "ucl": 3.4,   "unit": "g/L", "name": "에칭 첨가제"},
}


def prepare_features(lot_data):
    row = {}
    for csv_col, feat in CONTINUOUS_MAP.items():
        v = lot_data.get(csv_col, np.nan)
        row[feat] = float(v) if v is not None and str(v) not in ("", "nan", "None") else np.nan
    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw = str(lot_data.get(csv_col, "")).strip()
        norm = raw.replace(" ", "_") if cfg["prefix"] == "rework_history" else raw
        for v in cfg["values"]:
            nv = v.replace(" ", "_") if cfg["prefix"] == "rework_history" else v
            row[f"{cfg[\"prefix\"]}{cfg[\"sep\"]}{nv}"] = 1 if norm == nv else 0
    for feat in MODEL_FEATURES:
        if feat not in row:
            row[feat] = 0
    return pd.DataFrame([row])[MODEL_FEATURES]


def check_opls(X):
    alerts = []
    for feat, lim in OPLS_BOUNDS.items():
        if feat not in X.columns:
            continue
        val = X[feat].iloc[0]
        if pd.isna(val):
            continue
        if val > lim["ucl"]:   status = "UCL 초과"
        elif val < lim["lcl"]: status = "LCL 미달"
        else:                  status = "정상"
        alerts.append({"항목": lim["name"], "현재값": round(val,4), "LCL": lim["lcl"],
                        "SL": lim["sl"], "UCL": lim["ucl"], "단위": lim["unit"], "상태": status})
    return alerts


def ask_claude(question, lot_data, pred, X, alerts, api_key):
    client  = anthropic.Anthropic(api_key=api_key)
    rag_ctx = "\n---\n".join(json.dumps(r, ensure_ascii=False, indent=2) for r in paper_rules[:3])
    if shap_content:
        rag_ctx += "\n\n[SHAP 분석]\n" + shap_content[:1500]

    alert_lines = [f"  - {a[\"항목\"]}: {a[\"현재값\"]}{a[\"단위\"]} [{a[\"상태\"]}]" for a in alerts if a["상태"] != "정상"]
    lot_summary = f"""
LOT: {lot_data.get(\"LOT\")} | 제품: {lot_data.get(\"통합코드\")} | 검사결과: {lot_data.get(\"Result Ng2\")}
예측 에칭 속도: {pred:.4f} m/min  실제: {lot_data.get(\"부식 Speed\")} m/min
OPLS 이탈: {chr(10).join(alert_lines) if alert_lines else \"없음\"}
에칭 온도: {X[\"meas_etch_temp\"].iloc[0]:.1f}°C  에칭 비중: {X[\"meas_etch_sg\"].iloc[0]:.4f}  Cu: {X[\"meas_etch_cu\"].iloc[0]:.1f}g/L
"""
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1500,
        system=f"""당신은 PCB 에칭 공정 전문가 AI입니다. 공정 지식베이스를 참고하여 답변하세요.\n\n{rag_ctx}""",
        messages=[{"role": "user", "content": f"LOT 현황:\n{lot_summary}\n\n질문: {question}"}],
    )
    return resp.content[0].text


# ── Streamlit UI ───────────────────────────────────────────────────────────────
st.set_page_config(page_title="PRAGma — 에칭 공정 분석", page_icon="⚙️", layout="wide")
st.title("⚙️ PRAGma — 에칭 공정 MES 연동 분석")
st.caption("LOT 번호 입력 → MES 공정 데이터 자동 조회 → LightGBM 예측 → Claude LLM 해석")

with st.sidebar:
    st.header("설정")
    api_key = st.text_input("Anthropic API Key", value=os.environ.get("ANTHROPIC_API_KEY", ""), type="password")
    st.divider()
    st.caption(f"모델 피처 수: {len(MODEL_FEATURES)}")
    st.caption(f"MES LOT 수: {len(df_mes)}")
    st.caption(f"지식베이스 rule 수: {len(paper_rules)}")

col1, col2 = st.columns([1, 2])

with col1:
    lotno    = st.text_input("LOT 번호", value="A20000", placeholder="예: A20000")
    question = st.text_area("질문", value="이 lot의 에칭 공정 상태를 분석하고 개선 방향을 알려주세요.", height=100)
    run_btn  = st.button("분석 시작", type="primary", use_container_width=True)

if run_btn:
    if not api_key:
        st.error("사이드바에 Anthropic API Key를 입력하세요.")
        st.stop()

    rows = df_mes[df_mes["LOT"] == lotno]
    if rows.empty:
        st.error(f"LOT \'{lotno}\' 를 찾을 수 없습니다.")
        st.stop()

    lot_data = rows.iloc[0].to_dict()
    X        = prepare_features(lot_data)
    pred     = float(lgbm_model.predict(X)[0])
    actual   = lot_data.get("부식 Speed")
    alerts   = check_opls(X)

    with col2:
        st.subheader(f"LOT {lotno} 분석 결과")

        m1, m2, m3 = st.columns(3)
        m1.metric("예측 에칭 속도", f"{pred:.4f} m/min")
        m2.metric("실제 에칭 속도", f"{actual} m/min",
                  delta=f"{pred-actual:+.4f}" if isinstance(actual, (int, float)) else None)
        m3.metric("검사 결과", lot_data.get("Result Ng2", "N/A"))

        st.divider()
        st.subheader("OPLS 공정 기준 상태")
        df_alerts = pd.DataFrame(alerts)
        def color_status(v):
            if v != "정상":
                return "background-color: #ffcccc"
            return "background-color: #ccffcc"
        st.dataframe(df_alerts.style.applymap(color_status, subset=["상태"]), use_container_width=True)

        st.divider()
        with st.spinner("Claude LLM 분석 중..."):
            answer = ask_claude(question, lot_data, pred, X, alerts, api_key)
        st.subheader("AI 공정 분석")
        st.markdown(answer)
'''

out_path = BASE_DIR / "pragma_streamlit.py"
out_path.write_text(STREAMLIT_CODE.strip(), encoding="utf-8")
print(f"Streamlit 앱 저장 완료: {out_path}")
print(f"\n실행 명령어:")
print(f"  streamlit run {out_path}")